# Data audit

The agent answers customers from these CSVs, and the store manual says prices, stock and deadlines must never be stated without checking the system. This notebook checks whether the data can be trusted as-is.

Every finding ends with a **decision** that the agent's tools implement. There is no model training here, so there is no feature engineering: the few derived values the agent needs (final price, availability, days since delivery) are computed in the tools at question time.

In [1]:
import json
from pathlib import Path

import pandas as pd

DATA = Path("../data")
TODAY = pd.Timestamp.today().normalize()


def load(name: str) -> pd.DataFrame:
    return pd.read_csv(DATA / f"desafio_tecnico_ai_eng - {name}.csv")


categories = load("categories")
customers = load("customers")
products = load("products")
promotions = load("promotions")
orders = load("orders")
order_items = load("order_items")

orders["order_date"] = pd.to_datetime(orders["order_date"])
orders["estimated_delivery"] = pd.to_datetime(orders["estimated_delivery"])

tables = {
    "categories": categories,
    "customers": customers,
    "products": products,
    "promotions": promotions,
    "orders": orders,
    "order_items": order_items,
}
pd.DataFrame({name: df.shape for name, df in tables.items()}, index=["rows", "columns"])

,categories,customers,products,promotions,orders,order_items
rows,9,50,65,25,20,22
columns,3,5,9,5,9,3


## 1. Types and the `specs` column

In [2]:
pd.concat({name: df.dtypes.astype(str) for name, df in tables.items()}).to_frame("dtype")

dtype
categories  category_id                  int64
            name                           str
            description                    str
customers   customer_id                  int64
            name                           str
            phone                          str
            email                          str
            city                           str
products    product_id                   int64
            price_brl                  float64
            name                           str
            category_id                  int64
            description                    str
            stock_quantity               int64
            status                         str
            specs                          str
            created_at                     str
promotions  promotion_id                 int64
            product_id                   int64
            discount_percent             int64
            description                    str
            is_active                    int64
orders      order_id                     int64
            customer_id                  int64
            order_date          datetime64[us]
            status                         str
            total_brl                  float64
            payment_method                 str
            tracking_code                  str
            estimated_delivery  datetime64[us]
            notes                          str
order_items order_id                     int64
            quantity                     int64
            product_id                   int64

In [3]:
specs = products["specs"].map(json.loads)
pd.Series([key for spec in specs for key in spec]).value_counts().to_frame("products_with_key")

,products_with_key
color,65
neck,59
strings,59
top,50
back_sides,50
scale,50
electronics,50
body,9
shells,3
pieces,3


In [4]:
spec_keys = pd.DataFrame({
    "category": products["category_id"].map(categories.set_index("category_id")["name"]),
    "key": specs.map(list),
}).explode("key")
pd.crosstab(spec_keys["key"], spec_keys["category"])

category,Baixos,Baterias e Percussão,Guitarras,Teclados e Pianos,Ukuleles,Violões
key,,,,,,
back_sides,0,0,0,0,15,35
body,4,0,5,0,0,0
color,4,3,5,3,15,35
electronics,0,0,0,0,15,35
hardware,0,3,0,0,0,0
keys,0,0,0,3,0,0
neck,4,0,5,0,15,35
pieces,0,3,0,0,0,0
polyphony,0,0,0,3,0,0


**Finding.** Prices, stock and IDs load as numbers with no cleaning. Every `specs` value is valid JSON, but the keys depend on the product category: `top`, `back_sides`, `scale` and `electronics` appear only for violões and ukuleles; `body` for guitars and basses; `shells`, `pieces` and `hardware` for drums; `keys`, `type` and `polyphony` for keyboards. Only `color` is in every product.

**Decision.** Keep `specs` as a dictionary and return it as-is inside product results. Flattening it into columns would create mostly empty fields.

## 2. Missing values

In [5]:
missing = pd.concat({name: df.isna().sum() for name, df in tables.items()})
missing[missing > 0].to_frame("missing")

missing
orders tracking_code            12
       estimated_delivery       12
       notes                    16

In [6]:
orders.groupby("status").agg(
    orders=("order_id", "count"),
    with_tracking=("tracking_code", "count"),
    with_delivery_date=("estimated_delivery", "count"),
    with_notes=("notes", "count"),
)

,orders,with_tracking,with_delivery_date,with_notes
status,,,,
cancelled,4,0,0,4
confirmed,4,0,0,0
delivered,7,7,7,0
pending,4,0,0,0
shipped,1,1,1,0


In [7]:
orders.loc[orders["notes"].notna(), ["order_id", "status", "notes"]]

,order_id,status,notes
16,17,cancelled,Pedido cancelado pelo cliente antes do process...
17,18,cancelled,Item fora de estoque no momento do pedido
18,19,cancelled,Pagamento não confirmado dentro do prazo
19,20,cancelled,Endereço de entrega inválido informado pelo cl...


**Finding.** Gaps exist only in `orders`. Tracking code and estimated delivery exist exactly for shipped and delivered orders, and `notes` exists only for cancelled ones, where it holds the reason.

**Decision.** A missing tracking code means "not shipped yet", not "unknown". For cancelled orders the agent can quote the reason from `notes`.

## 3. Categorical values

In [8]:
for column, values in {
    "products.status": products["status"],
    "orders.status": orders["status"],
    "orders.payment_method": orders["payment_method"],
    "promotions.is_active": promotions["is_active"],
}.items():
    print(f"{column}: {values.value_counts().to_dict()}")

products.status: {'active': 62, 'discontinued': 2, 'coming_soon': 1}
orders.status: {'delivered': 7, 'confirmed': 4, 'pending': 4, 'cancelled': 4, 'shipped': 1}
orders.payment_method: {'pix': 5, 'credit_12x': 4, 'credit_6x': 3, 'boleto': 3, 'credit_3x': 3, 'debit': 2}
promotions.is_active: {0: 21, 1: 4}


**Finding.** Status and payment fields are English codes (`credit_6x`, `shipped`), while the store manual is in Portuguese.

**Decision.** The tools translate these codes to Portuguese labels, so the agent never shows a raw code or guesses its meaning.

## 4. Referential integrity

In [9]:
pd.Series({
    "order_items.product_id exists in products": order_items["product_id"].isin(products["product_id"]).all(),
    "order_items.order_id exists in orders": order_items["order_id"].isin(orders["order_id"]).all(),
    "every order has at least one item": orders["order_id"].isin(order_items["order_id"]).all(),
    "orders.customer_id exists in customers": orders["customer_id"].isin(customers["customer_id"]).all(),
    "promotions.product_id exists in products": promotions["product_id"].isin(products["product_id"]).all(),
    "products.category_id exists in categories": products["category_id"].isin(categories["category_id"]).all(),
}).to_frame("ok")

,ok
order_items.product_id exists in products,True
order_items.order_id exists in orders,True
every order has at least one item,True
orders.customer_id exists in customers,True
promotions.product_id exists in products,True
products.category_id exists in categories,True


**Finding.** Every reference resolves.

**Decision.** Joins between tables need no fallback for missing rows.

## 5. Duplicates

In [10]:
print("Duplicated product names:", products.duplicated("name").sum())
print("Duplicated customer phones:", customers.duplicated("phone").sum())

orders_per_customer = orders.groupby("customer_id").size().rename("orders")
(
    customers[customers.duplicated("email", keep=False)]
    .join(orders_per_customer, on="customer_id")
    .fillna({"orders": 0})
)

Duplicated product names: 0
Duplicated customer phones: 0


,customer_id,name,phone,email,city,orders
10,11,Bruno Carvalho Martins,(67) 98543-2109,bruno.martins@jmail.com,Três Lagoas,1.0
42,43,Bruno Martins,(67) 99456-3456,bruno.martins@jmail.com,Três Lagoas,0.0


**Finding.** Two customer records (11 and 43) share the e-mail `bruno.martins@jmail.com`, with different names and phones. Phones are unique.

**Decision.** E-mail alone does not identify one customer. Order lookups require an e-mail or phone that belongs to the customer on the order, and when an e-mail matches more than one record the agent asks for the phone number, which is unique.

## 6. Catalog coverage

In [11]:
(
    categories.merge(products, on="category_id", how="left", suffixes=("_category", ""))
    .groupby("name_category")
    .agg(
        products=("product_id", "count"),
        min_price=("price_brl", "min"),
        max_price=("price_brl", "max"),
        units_in_stock=("stock_quantity", "sum"),
    )
    .sort_values("products", ascending=False)
)

,products,min_price,max_price,units_in_stock
name_category,,,,
Violões,35,429.9,12000.0,198.0
Ukuleles,15,159.9,1899.0,99.0
Guitarras,5,5400.0,18900.0,42.0
Baixos,4,441.0,11445.0,42.0
Baterias e Percussão,3,4439.0,14922.0,23.0
Teclados e Pianos,3,1127.0,19567.0,13.0
Cordas Orquestrais,0,NaN,NaN,0.0
Instrumentos de Sopro (Madeiras),0,NaN,NaN,0.0
Instrumentos de Sopro (Metais),0,NaN,NaN,0.0


**Finding.** The catalog has 65 products, mostly acoustic guitars (35) and ukuleles (15). Three categories (woodwinds, brass, orchestral strings) have no products, even though the manual (section 1) describes a catalog of 300+ instruments that includes them.

**Decision.** The CSVs are the source of truth. For an empty category, the agent says the item is not available in the system and does not invent one.

## 7. Product status vs stock

In [12]:
products.loc[
    (products["status"] != "active") | (products["stock_quantity"] == 0),
    ["product_id", "name", "status", "stock_quantity", "price_brl"],
]

,product_id,name,status,stock_quantity,price_brl
15,96,Giannini GF-3D Dreadnought Sunburst,active,0,799.9
32,113,Shelby SN-7C 7 Cordas Nylon Natural,discontinued,0,749.0
49,130,Kala KA-ABP-CTG Barítono Cedar,coming_soon,2,1749.0
56,136,Ibanez Bass 2X Electric Bass,discontinued,13,441.0


In [13]:
(
    order_items.merge(products[["product_id", "name", "status"]], on="product_id")
    .merge(orders[["order_id", "status", "order_date"]], on="order_id", suffixes=("_product", "_order"))
    .query("status_product != 'active'")
)

,order_id,quantity,product_id,name,status_product,status_order,order_date
2,3,2,130,Kala KA-ABP-CTG Barítono Cedar,coming_soon,delivered,2025-11-20
13,14,1,130,Kala KA-ABP-CTG Barítono Cedar,coming_soon,pending,2026-03-15


**Finding.** Status and stock disagree in several places:
- product 96 is active with zero stock
- product 136 is discontinued with 13 units in stock
- product 130 is "coming soon", yet it appears in a delivered order (3) and a pending one (14)

**Decision.** A product is available only when it is `active` and has stock. Status wins over stock: discontinued products get equivalent suggestions, as the manual asks, and "coming soon" products cannot be bought yet. Existing orders are reported exactly as recorded.

## 8. Promotions

In [14]:
promo = promotions.merge(
    products[["product_id", "name", "price_brl", "status", "stock_quantity"]], on="product_id"
).assign(promo_price=lambda d: (d["price_brl"] * (1 - d["discount_percent"] / 100)).round(2))

promo[promo["is_active"] == 1]

,promotion_id,product_id,discount_percent,description,is_active,name,price_brl,status,stock_quantity,promo_price
11,12,127,10,Oferta Especial,1,Kalani KAL-700T Tenor Natural,359.0,active,11,323.10
13,14,121,20,Desconto Exclusivo,1,Ohana CK-20 Concert Natural,549.0,active,6,439.20
18,19,90,18,Desconto Exclusivo,1,Crafter HT-100 Folk Aço Natural,2399.0,active,3,1967.18
23,24,94,8,Desconto Exclusivo,1,Taylor 110e Dreadnought Natural,5999.0,active,4,5519.08


In [15]:
promo.groupby("product_id").filter(lambda g: len(g) > 1)[
    ["product_id", "name", "discount_percent", "description", "is_active"]
]

,product_id,name,discount_percent,description,is_active
5,93,Martin D-28 Dreadnought Natural,15,Promoção de Páscoa,0
7,96,Giannini GF-3D Dreadnought Sunburst,18,Semana do Músico,0
8,99,Tagima Classical Elétrico Nylon Sunburst,8,Promoção de Páscoa,0
21,96,Giannini GF-3D Dreadnought Sunburst,40,Semana do Músico,0
22,93,Martin D-28 Dreadnought Natural,10,Promoção de Páscoa,0
24,99,Tagima Classical Elétrico Nylon Sunburst,14,Aniversário da Loja,0


In [16]:
promo[promo["name"].str.contains("GD20")]

,promotion_id,product_id,discount_percent,description,is_active,name,price_brl,status,stock_quantity,promo_price
3,4,95,12,Oferta Especial,0,Takamine GD20 Dreadnought Natural,2199.0,active,5,1935.12


**Finding.**
- Only 4 of the 25 promotions are active.
- Products 93, 96 and 99 have two promotion rows each, all inactive.
- The Takamine GD20 promotion (12%) is inactive.
- Promotions have no dates, only the `is_active` flag.

**Decision.** Only promotions with `is_active = 1` apply. Promotions do not stack (manual 6.2) and no product has two active promotions, so no tie-break is needed. The 5% PIX discount applies only to products with no active promotion (manual 6.2). Answers always show the original price, the discount and the final price, as the manual requires. An inactive promotion is reported as ended, with the current price.

## 9. Order totals vs item prices

In [17]:
items_total = (
    order_items.merge(products[["product_id", "price_brl"]], on="product_id")
    .assign(line_total=lambda d: d["quantity"] * d["price_brl"])
    .groupby("order_id")["line_total"]
    .sum()
    .rename("items_at_current_price")
)
totals = orders.set_index("order_id")[["status", "payment_method", "total_brl"]].join(items_total)
totals["difference"] = (totals["total_brl"] - totals["items_at_current_price"]).round(2)

print(f"{(totals['difference'] == 0).sum()} of {len(totals)} orders match the current catalog price")
totals[totals["difference"] != 0]

18 of 20 orders match the current catalog price


,status,payment_method,total_brl,items_at_current_price,difference
order_id,,,,,
3,delivered,credit_12x,3450.0,3498.0,-48.0
20,cancelled,credit_3x,1400.0,1488.0,-88.0


In [18]:
mismatched = totals.index[totals["difference"] != 0]
order_items[order_items["order_id"].isin(mismatched)].merge(
    promotions[["product_id", "promotion_id", "discount_percent", "is_active"]], on="product_id", how="left"
)

,order_id,quantity,product_id,promotion_id,discount_percent,is_active
0,3,2,130,NaN,NaN,NaN
1,20,1,107,NaN,NaN,NaN
2,20,1,84,NaN,NaN,NaN


**Finding.** `total_brl` is in reais, the same unit as product prices: 18 of 20 orders match exactly. Two orders (3 and 20) have a recorded total R$ 48 and R$ 88 below their items at today's prices, and none of their products has a promotion row, so promotions do not explain it. Prices may have changed since the purchase, but the data cannot confirm that.

**Decision.** `total_brl` is the recorded total and is authoritative. The agent never recomputes an order total from current prices.

## 10. Dates relative to today

In [19]:
print("Today:", TODAY.date())
print("Orders placed between", orders["order_date"].min().date(), "and", orders["order_date"].max().date())

orders.assign(
    days_since_order=(TODAY - orders["order_date"]).dt.days,
    days_since_delivery=(TODAY - orders["estimated_delivery"]).dt.days,
).loc[
    orders["status"] != "cancelled",
    ["order_id", "status", "order_date", "estimated_delivery", "days_since_order", "days_since_delivery"],
]

Today: 2026-11-01
Orders placed between 2025-10-15 and 2026-03-22


,order_id,status,order_date,estimated_delivery,days_since_order,days_since_delivery
0,1,delivered,2025-10-15,2025-10-25,382,372.0
1,2,delivered,2025-11-02,2025-11-15,364,351.0
2,3,delivered,2025-11-20,2025-12-05,346,331.0
3,4,delivered,2025-12-10,2025-12-22,326,314.0
4,5,delivered,2026-01-05,2026-01-18,300,287.0
5,6,delivered,2026-01-18,2026-02-01,287,273.0
6,7,delivered,2026-02-03,2026-02-17,271,257.0
7,8,shipped,2026-02-15,2026-02-28,259,246.0
8,9,confirmed,2026-02-20,NaT,254,NaN
9,10,confirmed,2026-02-25,NaT,249,NaN


**Finding.**
- The table leaves out the 4 cancelled orders, which have no delivery.
- The data ends in March 2026. Every delivered order is past the 7-day regret window, the 30-day defect exchange and the 90-day legal warranty.
- Pending and confirmed orders have been open for months, and order 8 is still "shipped" months after its estimated delivery.
- There is no actual delivery date, only `estimated_delivery`.

**Decision.** The reference date is the real current date, because a customer service agent answers on the day the customer writes. The CSVs are a snapshot of the store system, so order status is reported as recorded. When a shipped order is past its estimated delivery, the agent says so and offers to forward the case to the team, as manual 7.3 does for complaints. Day counts are computed in code, never by the model, so policy deadlines are applied correctly. The receipt date is assumed to be `estimated_delivery`.

## Summary

| Finding | Decision |
|---|---|
| `specs` keys vary by instrument family | Keep as a dictionary, return as-is |
| Tracking and delivery date only for shipped and delivered orders | Missing tracking means "not shipped yet" |
| Status and payment fields are English codes | Tools translate them to Portuguese |
| Duplicate e-mail across two customers | Verify contact against the order's customer; ask for the phone when the e-mail is ambiguous |
| Woodwinds, brass and orchestral strings have no products | CSVs are the source of truth; never invent items |
| Status and stock disagree | Available = `active` and in stock; status wins |
| 4 active promotions; Takamine GD20 promotion ended | Apply only active ones; no PIX discount on top |
| Orders 3 and 20 totals differ from current prices | `total_brl` is authoritative |
| Data ends in March 2026; order 8 overdue; no actual delivery date | Real current date as reference; report status as recorded and flag overdue orders; day counts in code; receipt date = `estimated_delivery` |